<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/19_GES_Aware_Genomic_RAG_Cell_7C12_Blinded_Automated_Response_Level_Scoring_and_Freeze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact frozen lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import math
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '19_GES_Aware_Genomic_RAG_Cell_7C12_'
    'Blinded_Automated_Response_Level_Scoring_and_Freeze.ipynb'
)
CELL_ID = '7C12'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_RESPONSES_PER_QUESTION = 18
EXPECTED_CONTEXT_PACKETS = 5

PRIMARY_ENDPOINT_NAME = 'automated_evidence_fidelity_pass'

EXPECTED_CELL_7C11R_TERMINAL_DECISION = (
    'PASS_STAGE7C11R_A004_INPUT_COMPLETENESS_REMEDIATED_BEFORE_SCORING_'
    'CELL7C10_DETERMINISTIC_INPUT_V1_PRESERVED_CORRECTED_V2_1440_ROWS_WITH_'
    'EXPECTED_AGGREGATE_CONFLICT_FLAG_APPENDED_FROM_FROZEN_SCORE_BLIND_ANSWER_'
    'KEYS_A004_ENDPOINT_UNCHANGED_CELL7C12_BLINDED_AUTOMATED_SCORING_ONLY_'
    'AUTHORIZED_NO_HUMAN_REVIEW_CONDITION_UNBLINDING_ROUTING_ACCESS_RUN_'
    'AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

EXPECTED_CELL_7C11R_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C12_FULLY_AUTOMATED_BLINDED_STRUCTURED_EVALUATION_'
    'OF_1440_FROZEN_RESPONSES_USING_A004_ENDPOINT_SPEC_AND_CELL7C11R_CORRECTED_'
    'DETERMINISTIC_SCORING_INPUT_V2_WITH_EXPECTED_AGGREGATE_CONFLICT_FLAG_NO_'
    'HUMAN_REVIEW_CONDITION_UNBLINDING_INTERNAL_ROUTING_ACCESS_RUN_AGGREGATION_'
    'BOOTSTRAP_OR_ARM_COMPARISON'
)

# --------------------------------------------------------------------------------------
# Cell 7C11R package.
# --------------------------------------------------------------------------------------
CELL_7C11R_DATA_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c11r_a004_deterministic_input_remediation_v1'
)
CELL_7C11R_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c11r_a004_deterministic_input_remediation_v1'
)
CELL_7C11R_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c11r_a004_deterministic_input_remediation_v1'
)

CELL_7C11R = OrderedDict([
    ('corrected_deterministic_input', {
        'path': CELL_7C11R_DATA_DIR / 'cell_7c11r_a004_deterministic_scoring_input_v2.parquet',
        'sha256': 'ac55ab77d0a181cd811e3b6391efca969f1f91451fc70e76a8972ce75c73c598',
    }),
    ('remediation_authorization', {
        'path': CELL_7C11R_CONFIG_DIR / 'cell_7c11r_a004_input_completeness_remediation_authorization_v1.json',
        'sha256': 'b2707da3ad14c0b99eae267b39a811648c2a94d98c4d55e10f2eebc4170db1a7',
    }),
    ('input_inventory', {
        'path': CELL_7C11R_CONFIG_DIR / 'cell_7c11r_verified_input_inventory_v1.csv',
        'sha256': '6daa8315db6873ef623e4b942d5f24ea34d4399cf843be7e1752336b17d5d4f9',
    }),
    ('qc', {
        'path': CELL_7C11R_QC_DIR / 'cell_7c11r_a004_input_completeness_remediation_qc_v1.json',
        'sha256': '665963878839b2b9b08dfc62d12dba3fc2af8346d2cb87bc6bc5b07bbaf754e7',
    }),
    ('manifest', {
        'path': CELL_7C11R_CONFIG_DIR / 'cell_7c11r_a004_input_completeness_remediation_manifest_v1.json',
        'sha256': '630e5df5e9c766ab930ec5af11198457a24427b71954be50193fb6ac7df1d92d',
    }),
])

# --------------------------------------------------------------------------------------
# Exact frozen A004 endpoint spec from Cell 7C11.
# --------------------------------------------------------------------------------------
CELL_7C11_A004_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)

CELL_7C11_ENDPOINT_SPEC = {
    'path': CELL_7C11_A004_DIR / 'protocol_amendment_A004_automated_endpoint_spec_v1.json',
    'sha256': '40cf8c54be9aa0337240caa6f6c1f8a31d0baf3078c94638fe8bf79e212edb97',
}

# --------------------------------------------------------------------------------------
# Cell 7C12 outputs.
# --------------------------------------------------------------------------------------
OUT_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
)

OUTPUTS = OrderedDict([
    ('response_level_outcomes',
     OUT_DIR / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1.parquet'),
    ('endpoint_derivation_schema',
     CONFIG_DIR / 'cell_7c12_a004_endpoint_derivation_schema_v1.json'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c12_verified_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c12_a004_blinded_scoring_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c12_a004_blinded_scoring_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c12_a004_blinded_scoring_manifest_v1.json'),
])

for directory in (OUT_DIR, CONFIG_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C12 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Outcome directory: {OUT_DIR}')
print(f'Config directory : {CONFIG_DIR}')
print(f'QC directory     : {QC_DIR}')

Outcome directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage7_rag/cell_7c12_a004_blinded_automated_response_level_outcomes_v1
Config directory : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c12_a004_blinded_automated_response_level_outcomes_v1
QC directory     : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c12_a004_blinded_automated_response_level_outcomes_v1


## 2. SHA-256, stable serialization, and validation helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    frame.to_parquet(path, index=False, engine='pyarrow', compression='zstd')
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


def normalize_bool(value: Any, field_name: str, review_item_id: str) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value in (0, 1):
        return bool(value)
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {'true', '1'}:
            return True
        if normalized in {'false', '0'}:
            return False
    raise ValueError(
        f'Invalid Boolean in {field_name} for {review_item_id}: {value!r}'
    )


def parse_string_list(value: Any, field_name: str, review_item_id: str) -> list[str]:
    if isinstance(value, list):
        parsed = value
    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f'Invalid JSON in {field_name} for {review_item_id}: {exc}'
            ) from exc
    else:
        raise TypeError(
            f'{field_name} must be JSON string or list for {review_item_id}; got {type(value).__name__}'
        )

    if not isinstance(parsed, list):
        raise TypeError(f'{field_name} did not decode to a list for {review_item_id}.')

    normalized = []
    for item in parsed:
        if item is None:
            raise ValueError(f'Null evidence ID in {field_name} for {review_item_id}.')
        token = str(item).strip()
        if not token:
            raise ValueError(f'Blank evidence ID in {field_name} for {review_item_id}.')
        normalized.append(token)
    return normalized


def stable_unique(values: list[str]) -> list[str]:
    seen = set()
    out = []
    for value in values:
        if value not in seen:
            out.append(value)
            seen.add(value)
    return out


with tempfile.TemporaryDirectory(prefix='cell_7c12_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / parsing helper self-test: PASS')

Serialization / parsing helper self-test: PASS


## 3. Reverify Cell 7C11R authorization and the exact A004 endpoint specification

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C11R.items():
    record = verify_exact_artifact(
        f'cell_7c11r_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C11R'
    verified_inputs.append(record)

record = verify_exact_artifact(
    'cell_7c11_a004_endpoint_spec',
    CELL_7C11_ENDPOINT_SPEC['path'],
    CELL_7C11_ENDPOINT_SPEC['sha256'],
)
record['source_cell'] = '7C11'
verified_inputs.append(record)

authorization_7c11r = load_json(CELL_7C11R['remediation_authorization']['path'])
manifest_7c11r = load_json(CELL_7C11R['manifest']['path'])
qc_7c11r = load_json(CELL_7C11R['qc']['path'])
endpoint_spec = load_json(CELL_7C11_ENDPOINT_SPEC['path'])

if manifest_7c11r.get('terminal_decision') != EXPECTED_CELL_7C11R_TERMINAL_DECISION:
    raise AssertionError('Cell 7C11R terminal PASS mismatch.')
if authorization_7c11r.get('authorization_decision') != EXPECTED_CELL_7C11R_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C11R authorization decision mismatch.')
if manifest_7c11r.get('next_authorized_cell') != '7C12':
    raise AssertionError('Cell 7C11R does not authorize Cell 7C12.')
if manifest_7c11r.get('condition_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C11R unexpectedly authorizes condition unblinding.')
if manifest_7c11r.get('internal_routing_map_access_authorized') is not False:
    raise AssertionError('Cell 7C11R unexpectedly authorizes routing-map access.')
if manifest_7c11r.get('run_aggregation_authorized') is not False:
    raise AssertionError('Cell 7C11R unexpectedly authorizes run aggregation.')
if manifest_7c11r.get('bootstrap_inference_authorized') is not False:
    raise AssertionError('Cell 7C11R unexpectedly authorizes bootstrap inference.')
if int(qc_7c11r.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C11R QC does not report zero failures.')

primary = endpoint_spec['primary_automated_endpoint']
if primary.get('name') != PRIMARY_ENDPOINT_NAME:
    raise AssertionError('A004 primary endpoint name mismatch.')
if set(primary.get('components', {}).keys()) != {
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
}:
    raise AssertionError('A004 primary component set mismatch.')

print('Cell 7C11R package                    : 5/5 exact hashes + sidecars')
print('Cell 7C11 A004 endpoint spec           : exact hash + sidecar verified')
print('Cell 7C12 authorization                : VERIFIED')
print('A004 primary endpoint                  : VERIFIED')
print('Condition unblinding                   : PROHIBITED')
print('Routing-map access                     : PROHIBITED')

Cell 7C11R package                    : 5/5 exact hashes + sidecars
Cell 7C11 A004 endpoint spec           : exact hash + sidecar verified
Cell 7C12 authorization                : VERIFIED
A004 primary endpoint                  : VERIFIED
Condition unblinding                   : PROHIBITED
Routing-map access                     : PROHIBITED


## 4. Load and structurally validate the corrected score-blind deterministic input

In [5]:
scoring_input = pd.read_parquet(
    CELL_7C11R['corrected_deterministic_input']['path']
)

REQUIRED_COLUMNS = [
    'review_item_id',
    'question_id',
    'response_policy',
    'conflict_detected',
    'evidence_strength',
    'confidence',
    'cited_evidence_ids_json',
    'context_packet_ids_json',
    'expected_abstention_or_qualification_required',
    'deterministic_scores_calculated',
    'expected_aggregate_conflict_flag',
]

missing = [column for column in REQUIRED_COLUMNS if column not in scoring_input.columns]
if missing:
    raise AssertionError(
        'Corrected deterministic input is missing required field(s): '
        + ', '.join(missing)
    )

if len(scoring_input) != EXPECTED_RESPONSES:
    raise AssertionError(
        f'Expected {EXPECTED_RESPONSES:,} rows; observed {len(scoring_input):,}.'
    )
if scoring_input['review_item_id'].duplicated().any():
    raise AssertionError('Duplicate review_item_id detected.')
if scoring_input['question_id'].isna().any():
    raise AssertionError('Missing question_id detected.')

question_counts = scoring_input.groupby('question_id', dropna=False).size()
if len(question_counts) != EXPECTED_QUESTIONS:
    raise AssertionError(f'Expected 80 questions; observed {len(question_counts)}.')
if not question_counts.eq(EXPECTED_RESPONSES_PER_QUESTION).all():
    raise AssertionError('Every question must have exactly 18 frozen responses.')

# Strong score-blind schema check.
for prohibited in [
    'blinded_alias',
    'run_id',
    'generation_request_id',
    'prompt_instance_id',
    'condition_id',
    'condition_name',
    'quality_score',
    'quality_rank',
    'rrf_score',
    'semantic_rank',
    'semantic_score',
    'full_ges_p_stable_t1',
    'no_star_ges_p_stable_t1',
]:
    if prohibited in scoring_input.columns:
        raise AssertionError(f'Scoring input exposes prohibited field: {prohibited}')

allowed_policies = {'answer', 'cautious_answer', 'abstain'}
observed_policies = set(scoring_input['response_policy'].astype(str))
if not observed_policies.issubset(allowed_policies):
    raise AssertionError(
        f'Unexpected response_policy values: {sorted(observed_policies - allowed_policies)}'
    )

allowed_strengths = {'high', 'moderate', 'low'}
observed_strengths = set(scoring_input['evidence_strength'].astype(str))
if not observed_strengths.issubset(allowed_strengths):
    raise AssertionError(
        f'Unexpected evidence_strength values: {sorted(observed_strengths - allowed_strengths)}'
    )

confidence = pd.to_numeric(scoring_input['confidence'], errors='coerce')
if confidence.isna().any():
    raise AssertionError('Missing/non-numeric confidence values detected.')
if not confidence.between(0.0, 1.0, inclusive='both').all():
    raise AssertionError('Confidence value outside [0,1].')

if not scoring_input['deterministic_scores_calculated'].eq(False).all():
    raise AssertionError('Input indicates deterministic scores were already calculated.')

print(f'Score-blind input rows                 : {len(scoring_input):,}')
print(f'Primary questions                      : {len(question_counts)}')
print('Responses per question                 : 18')
print('Response policy domain                 : VERIFIED')
print('Evidence strength domain               : VERIFIED')
print('Confidence range                       : VERIFIED [0,1]')
print('Condition/routing fields present        : NO')

Score-blind input rows                 : 1,440
Primary questions                      : 80
Responses per question                 : 18
Response policy domain                 : VERIFIED
Evidence strength domain               : VERIFIED
Confidence range                       : VERIFIED [0,1]
Condition/routing fields present        : NO


## 5. Calculate the frozen A004 response-level automated outcomes

In [6]:
outcome_rows = []

for row in scoring_input.itertuples(index=False):
    review_item_id = str(row.review_item_id)
    question_id = str(row.question_id)
    response_policy = str(row.response_policy)
    evidence_strength = str(row.evidence_strength)
    confidence_value = float(row.confidence)

    conflict_detected = normalize_bool(
        row.conflict_detected,
        'conflict_detected',
        review_item_id,
    )
    expected_conflict = normalize_bool(
        row.expected_aggregate_conflict_flag,
        'expected_aggregate_conflict_flag',
        review_item_id,
    )
    expected_caution = normalize_bool(
        row.expected_abstention_or_qualification_required,
        'expected_abstention_or_qualification_required',
        review_item_id,
    )

    cited_ids = parse_string_list(
        row.cited_evidence_ids_json,
        'cited_evidence_ids_json',
        review_item_id,
    )
    context_ids = parse_string_list(
        row.context_packet_ids_json,
        'context_packet_ids_json',
        review_item_id,
    )

    if len(context_ids) != EXPECTED_CONTEXT_PACKETS:
        raise AssertionError(
            f'{review_item_id} has {len(context_ids)} context IDs; expected exactly 5.'
        )
    if len(set(context_ids)) != EXPECTED_CONTEXT_PACKETS:
        raise AssertionError(
            f'{review_item_id} context packet IDs are not exactly five unique values.'
        )

    context_set = set(context_ids)
    cited_unique = stable_unique(cited_ids)
    valid_cited_raw = [evidence_id for evidence_id in cited_ids if evidence_id in context_set]
    invalid_cited_raw = [evidence_id for evidence_id in cited_ids if evidence_id not in context_set]
    valid_cited_unique = stable_unique(valid_cited_raw)
    invalid_cited_unique = stable_unique(invalid_cited_raw)

    cited_count = len(cited_ids)
    distinct_cited_count = len(cited_unique)
    duplicate_cited_count = cited_count - distinct_cited_count

    valid_raw_count = len(valid_cited_raw)
    valid_distinct_count = len(valid_cited_unique)
    invalid_raw_count = len(invalid_cited_raw)
    invalid_distinct_count = len(invalid_cited_unique)

    citation_integrity_pass = (invalid_raw_count == 0)

    if response_policy in {'answer', 'cautious_answer'}:
        citation_presence_pass = valid_distinct_count >= 1
    elif response_policy == 'abstain':
        citation_presence_pass = True
    else:
        raise AssertionError(f'Unexpected response_policy for {review_item_id}: {response_policy}')

    conflict_concordance_pass = (conflict_detected == expected_conflict)

    if expected_caution:
        caution_policy_concordance_pass = response_policy in {'cautious_answer', 'abstain'}
        required_caution_compliance = caution_policy_concordance_pass
        over_abstention = pd.NA
    else:
        caution_policy_concordance_pass = response_policy in {'answer', 'cautious_answer'}
        required_caution_compliance = pd.NA
        over_abstention = (response_policy == 'abstain')

    automated_evidence_fidelity_pass = (
        citation_integrity_pass
        and citation_presence_pass
        and conflict_concordance_pass
        and caution_policy_concordance_pass
    )

    context_citation_coverage = valid_distinct_count / EXPECTED_CONTEXT_PACKETS

    outcome_rows.append({
        'review_item_id': review_item_id,
        'question_id': question_id,

        # Preserved descriptive structured response fields.
        'response_policy': response_policy,
        'evidence_strength': evidence_strength,
        'confidence': confidence_value,

        # Frozen targets needed for transparent audit.
        'expected_aggregate_conflict_flag': expected_conflict,
        'expected_abstention_or_qualification_required': expected_caution,
        'conflict_detected': conflict_detected,

        # Citation derivation fields.
        'cited_evidence_id_count': cited_count,
        'distinct_cited_evidence_id_count': distinct_cited_count,
        'duplicate_cited_evidence_id_count': duplicate_cited_count,
        'valid_context_evidence_id_count': valid_raw_count,
        'distinct_valid_context_evidence_id_count': valid_distinct_count,
        'invalid_or_hallucinated_evidence_id_count': invalid_raw_count,
        'distinct_invalid_or_hallucinated_evidence_id_count': invalid_distinct_count,
        'context_citation_coverage': float(context_citation_coverage),

        # Four frozen primary components.
        'citation_integrity_pass': bool(citation_integrity_pass),
        'citation_presence_pass': bool(citation_presence_pass),
        'conflict_concordance_pass': bool(conflict_concordance_pass),
        'caution_policy_concordance_pass': bool(caution_policy_concordance_pass),

        # Prespecified secondary policy diagnostics.
        'required_caution_compliance': required_caution_compliance,
        'over_abstention': over_abstention,

        # Primary A004 automated endpoint.
        PRIMARY_ENDPOINT_NAME: bool(automated_evidence_fidelity_pass),
    })

outcomes = pd.DataFrame(outcome_rows)

# Use explicit nullable Boolean dtype for conditionally applicable secondary fields.
outcomes['required_caution_compliance'] = outcomes['required_caution_compliance'].astype('boolean')
outcomes['over_abstention'] = outcomes['over_abstention'].astype('boolean')

if len(outcomes) != EXPECTED_RESPONSES:
    raise AssertionError('Outcome table must contain exactly 1,440 rows.')
if outcomes['review_item_id'].nunique() != EXPECTED_RESPONSES:
    raise AssertionError('Outcome review_item_id must be unique.')

primary_components = [
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
]
reconstructed_primary = outcomes[primary_components].all(axis=1)
if not reconstructed_primary.equals(outcomes[PRIMARY_ENDPOINT_NAME]):
    raise AssertionError('Primary composite does not exactly reconstruct from its four frozen components.')

if not outcomes['context_citation_coverage'].between(0.0, 1.0, inclusive='both').all():
    raise AssertionError('Context citation coverage outside [0,1].')

# Do not print scientific pass-rate summaries here.
print(f'Response-level outcomes calculated     : {len(outcomes):,}')
print('Primary components calculated          : 4 / 4')
print('Primary composite reconstruction        : EXACT')
print('Secondary automated fields              : MATERIALIZED')
print('Arm-level results inspected             : NO')

Response-level outcomes calculated     : 1,440
Primary components calculated          : 4 / 4
Primary composite reconstruction        : EXACT
Secondary automated fields              : MATERIALIZED
Arm-level results inspected             : NO


## 6. QC the blinded outcome table without inspecting condition performance

In [7]:
OUTCOME_COLUMNS = [
    'review_item_id',
    'question_id',
    'response_policy',
    'evidence_strength',
    'confidence',
    'expected_aggregate_conflict_flag',
    'expected_abstention_or_qualification_required',
    'conflict_detected',
    'cited_evidence_id_count',
    'distinct_cited_evidence_id_count',
    'duplicate_cited_evidence_id_count',
    'valid_context_evidence_id_count',
    'distinct_valid_context_evidence_id_count',
    'invalid_or_hallucinated_evidence_id_count',
    'distinct_invalid_or_hallucinated_evidence_id_count',
    'context_citation_coverage',
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
    'required_caution_compliance',
    'over_abstention',
    PRIMARY_ENDPOINT_NAME,
]

outcomes = outcomes[OUTCOME_COLUMNS].copy()

# Strong prohibition: outcome package must remain condition-blind.
for prohibited in [
    'blinded_alias',
    'run_id',
    'generation_request_id',
    'prompt_instance_id',
    'condition_id',
    'condition_name',
    'quality_score',
    'quality_rank',
    'rrf_score',
    'semantic_rank',
    'semantic_score',
    'full_ges_p_stable_t1',
    'no_star_ges_p_stable_t1',
]:
    if prohibited in outcomes.columns:
        raise AssertionError(f'Outcome table exposes prohibited field: {prohibited}')

qc_checks = OrderedDict([
    ('rows_1440', len(outcomes) == 1440),
    ('unique_review_items_1440', outcomes['review_item_id'].nunique() == 1440),
    ('questions_80', outcomes['question_id'].nunique() == 80),
    ('responses_per_question_18',
     outcomes.groupby('question_id').size().eq(18).all()),
    ('confidence_complete', outcomes['confidence'].notna().all()),
    ('confidence_range',
     outcomes['confidence'].between(0, 1, inclusive='both').all()),
    ('coverage_complete', outcomes['context_citation_coverage'].notna().all()),
    ('coverage_range',
     outcomes['context_citation_coverage'].between(0, 1, inclusive='both').all()),
    ('citation_counts_nonnegative',
     (outcomes[
         [
             'cited_evidence_id_count',
             'distinct_cited_evidence_id_count',
             'duplicate_cited_evidence_id_count',
             'valid_context_evidence_id_count',
             'distinct_valid_context_evidence_id_count',
             'invalid_or_hallucinated_evidence_id_count',
             'distinct_invalid_or_hallucinated_evidence_id_count',
         ]
     ] >= 0).all().all()),
    ('distinct_cited_le_raw',
     (outcomes['distinct_cited_evidence_id_count'] <= outcomes['cited_evidence_id_count']).all()),
    ('duplicate_count_identity',
     (
         outcomes['duplicate_cited_evidence_id_count']
         == outcomes['cited_evidence_id_count'] - outcomes['distinct_cited_evidence_id_count']
     ).all()),
    ('distinct_valid_le_5',
     outcomes['distinct_valid_context_evidence_id_count'].le(5).all()),
    ('coverage_identity',
     np.allclose(
         outcomes['context_citation_coverage'].to_numpy(dtype=float),
         outcomes['distinct_valid_context_evidence_id_count'].to_numpy(dtype=float) / 5.0,
         atol=0.0,
         rtol=0.0,
     )),
    ('citation_integrity_complete', outcomes['citation_integrity_pass'].notna().all()),
    ('citation_presence_complete', outcomes['citation_presence_pass'].notna().all()),
    ('conflict_concordance_complete', outcomes['conflict_concordance_pass'].notna().all()),
    ('caution_concordance_complete', outcomes['caution_policy_concordance_pass'].notna().all()),
    ('primary_complete', outcomes[PRIMARY_ENDPOINT_NAME].notna().all()),
    ('primary_exact_reconstruction',
     outcomes[
         [
             'citation_integrity_pass',
             'citation_presence_pass',
             'conflict_concordance_pass',
             'caution_policy_concordance_pass',
         ]
     ].all(axis=1).equals(outcomes[PRIMARY_ENDPOINT_NAME])),
    ('required_caution_applicability',
     outcomes.loc[
         outcomes['expected_abstention_or_qualification_required'],
         'required_caution_compliance'
     ].notna().all()),
    ('required_caution_na_when_not_required',
     outcomes.loc[
         ~outcomes['expected_abstention_or_qualification_required'],
         'required_caution_compliance'
     ].isna().all()),
    ('over_abstention_applicability',
     outcomes.loc[
         ~outcomes['expected_abstention_or_qualification_required'],
         'over_abstention'
     ].notna().all()),
    ('over_abstention_na_when_caution_required',
     outcomes.loc[
         outcomes['expected_abstention_or_qualification_required'],
         'over_abstention'
     ].isna().all()),
    ('no_condition_identity_columns',
     not any(
         field in outcomes.columns
         for field in ['blinded_alias', 'run_id', 'condition_id', 'condition_name']
     )),
    ('no_run_aggregation_performed', True),
    ('no_bootstrap_performed', True),
    ('no_arm_comparison_performed', True),
    ('no_human_review_performed', True),
    ('no_llm_call_performed', True),
])

failed = [name for name, passed in qc_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C12 blinded outcome QC failed:\\n- ' + '\\n- '.join(failed)
    )

print(f'Blinded outcome QC checks              : {len(qc_checks)}/{len(qc_checks)} PASS')
print('Condition identity columns              : ABSENT')
print('Run aggregation                         : NOT PERFORMED')
print('Bootstrap / arm comparison              : NOT PERFORMED')

Blinded outcome QC checks              : 29/29 PASS
Condition identity columns              : ABSENT
Run aggregation                         : NOT PERFORMED
Bootstrap / arm comparison              : NOT PERFORMED


## 7. Freeze the blinded response-level outcome package

In [8]:
endpoint_derivation_schema = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'created_utc': CREATED_UTC,
    'primary_endpoint': {
        'name': PRIMARY_ENDPOINT_NAME,
        'components': [
            'citation_integrity_pass',
            'citation_presence_pass',
            'conflict_concordance_pass',
            'caution_policy_concordance_pass',
        ],
        'rule':
            'logical AND of the four Boolean components',
    },
    'derivations': {
        'citation_integrity_pass':
            'invalid_or_hallucinated_evidence_id_count == 0',
        'citation_presence_pass':
            'if response_policy in {answer,cautious_answer}: '
            'distinct_valid_context_evidence_id_count >= 1; '
            'if abstain: True',
        'conflict_concordance_pass':
            'conflict_detected == expected_aggregate_conflict_flag',
        'caution_policy_concordance_pass':
            'if expected caution: response_policy in {cautious_answer,abstain}; '
            'else response_policy in {answer,cautious_answer}',
        'context_citation_coverage':
            'distinct_valid_context_evidence_id_count / 5',
        'required_caution_compliance':
            'same as caution_policy_concordance_pass only when expected caution is true; otherwise NA',
        'over_abstention':
            'response_policy == abstain only when expected caution is false; otherwise NA',
    },
    'frozen_context_packet_count': 5,
    'free_text_factual_correctness_scored': False,
    'semantic_citation_entailment_scored': False,
    'condition_identity_used': False,
    'run_id_used': False,
}

stable_write_parquet(OUTPUTS['response_level_outcomes'], outcomes)
write_sidecar(OUTPUTS['response_level_outcomes'])

stable_write_json(OUTPUTS['endpoint_derivation_schema'], endpoint_derivation_schema)
write_sidecar(OUTPUTS['endpoint_derivation_schema'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

terminal_decision = (
    'PASS_STAGE7C12_A004_BLINDED_AUTOMATED_RESPONSE_LEVEL_SCORING_COMPLETE_'
    '1440_OF_1440_FOUR_PRIMARY_COMPONENTS_AND_AUTOMATED_EVIDENCE_FIDELITY_'
    'COMPOSITE_FROZEN_CHECKSUM_PROTECTED_SCORE_BLIND_NO_HUMAN_REVIEW_FREE_TEXT_'
    'FACTUAL_CORRECTNESS_OR_SEMANTIC_CITATION_ENTAILMENT_CLAIM_NO_CONDITION_'
    'UNBLINDING_ROUTING_ACCESS_RUN_AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON_'
    'NEXT_AUTOMATED_EXECUTION_NOT_AUTHORIZED'
)

execution_report = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'input': {
        'corrected_deterministic_input_sha256':
            CELL_7C11R['corrected_deterministic_input']['sha256'],
        'A004_endpoint_spec_sha256': CELL_7C11_ENDPOINT_SPEC['sha256'],
        'rows_scored': EXPECTED_RESPONSES,
    },
    'scoring': {
        'primary_endpoint': PRIMARY_ENDPOINT_NAME,
        'primary_component_count': 4,
        'response_level_outcomes_complete': True,
        'free_text_factual_correctness_scored': False,
        'semantic_citation_entailment_scored': False,
    },
    'scientific_operations': {
        'condition_identity_unblinded': False,
        'internal_routing_map_opened': False,
        'run_aggregation_performed': False,
        'bootstrap_inference_performed': False,
        'arm_comparison_performed': False,
        'human_review_performed': False,
        'llm_called': False,
    },
    'next_required_action':
        'Separate authorization to reverify the frozen Cell 7C12 response-level outcomes and, only then, '
        'open the internal routing map for condition restoration and later question-level aggregation.',
    'terminal_decision': terminal_decision,
}

stable_write_json(OUTPUTS['execution_report'], execution_report)
write_sidecar(OUTPUTS['execution_report'])

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in qc_checks.items()},
    'passed_checks': len(qc_checks),
    'failed_checks': 0,
    'total_checks': len(qc_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c11r_manifest_sha256': CELL_7C11R['manifest']['sha256'],
        'cell_7c11r_corrected_deterministic_input_sha256':
            CELL_7C11R['corrected_deterministic_input']['sha256'],
        'cell_7c11_A004_endpoint_spec_sha256': CELL_7C11_ENDPOINT_SPEC['sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'response_level_outcomes_frozen': True,
    'condition_identity_unblinding_authorized': False,
    'internal_routing_map_access_authorized': False,
    'run_aggregation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
    'next_authorized_cell': None,
    'terminal_decision': terminal_decision,
}

stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh readback and exact structural verification.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C12 final readback failed: {path}')

rb = pd.read_parquet(OUTPUTS['response_level_outcomes'])
rb_schema = load_json(OUTPUTS['endpoint_derivation_schema'])
rb_report = load_json(OUTPUTS['execution_report'])
rb_qc = load_json(OUTPUTS['qc'])
rb_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('readback_rows_1440', len(rb) == 1440),
    ('readback_unique_items_1440', rb['review_item_id'].nunique() == 1440),
    ('readback_questions_80', rb['question_id'].nunique() == 80),
    ('readback_primary_present', PRIMARY_ENDPOINT_NAME in rb.columns),
    ('readback_primary_complete', rb[PRIMARY_ENDPOINT_NAME].notna().all()),
    ('readback_four_components_present',
     all(
         field in rb.columns
         for field in [
             'citation_integrity_pass',
             'citation_presence_pass',
             'conflict_concordance_pass',
             'caution_policy_concordance_pass',
         ]
     )),
    ('readback_primary_exact_reconstruction',
     rb[
         [
             'citation_integrity_pass',
             'citation_presence_pass',
             'conflict_concordance_pass',
             'caution_policy_concordance_pass',
         ]
     ].all(axis=1).equals(rb[PRIMARY_ENDPOINT_NAME])),
    ('schema_primary_exact', rb_schema['primary_endpoint']['name'] == PRIMARY_ENDPOINT_NAME),
    ('report_unblinding_false',
     rb_report['scientific_operations']['condition_identity_unblinded'] is False),
    ('report_routing_false',
     rb_report['scientific_operations']['internal_routing_map_opened'] is False),
    ('manifest_next_none', rb_manifest.get('next_authorized_cell') is None),
    ('manifest_unblinding_false',
     rb_manifest.get('condition_identity_unblinding_authorized') is False),
    ('manifest_routing_false',
     rb_manifest.get('internal_routing_map_access_authorized') is False),
    ('manifest_aggregation_false',
     rb_manifest.get('run_aggregation_authorized') is False),
    ('manifest_bootstrap_false',
     rb_manifest.get('bootstrap_inference_authorized') is False),
    ('manifest_arm_comparison_false',
     rb_manifest.get('arm_comparison_authorized') is False),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C12 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(qc_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C12')
print('A004 BLINDED AUTOMATED STRUCTURED RESPONSE-LEVEL SCORING AND FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM AUTHORIZATION')
print(f'Cell 7C11R manifest SHA-256                   : {CELL_7C11R["manifest"]["sha256"]}')
print('Cell 7C11R terminal PASS verified             : YES')
print('Corrected deterministic input V2              : exact SHA + sidecar verified')
print('A004 endpoint specification                   : exact SHA + sidecar verified')

print('\\nBLINDED RESPONSE-LEVEL SCORING')
print('Frozen response observations                  : 1,440')
print('Primary questions                             : 80')
print('Responses per question                        : 18')
print('Primary component 1                           : citation_integrity_pass')
print('Primary component 2                           : citation_presence_pass')
print('Primary component 3                           : conflict_concordance_pass')
print('Primary component 4                           : caution_policy_concordance_pass')
print('Primary composite                             : automated_evidence_fidelity_pass')
print('Primary composite reconstruction              : EXACT')

print('\\nSCOPE LIMITS')
print('Free-text factual correctness scored          : NO')
print('Semantic citation entailment scored           : NO')
print('Human review performed                        : NO')
print('Condition identity unblinded                  : NO')
print('Internal routing map opened                   : NO')
print('Run aggregation                               : NO')
print('Bootstrap / arm comparison                    : NO')

print('\\nCELL 7C12 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT BOUNDARY')
print('Blinded response-level automated outcomes     : FROZEN')
print('Next automated cell                           : NOT AUTHORIZED')
print('Required next step                            : separate unblinding/routing authorization')
print('Condition restoration / run aggregation       : STILL PROHIBITED')
print('Bootstrap / arm comparison                    : STILL PROHIBITED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C12
A004 BLINDED AUTOMATED STRUCTURED RESPONSE-LEVEL SCORING AND FREEZE
Notebook                                      : 19_GES_Aware_Genomic_RAG_Cell_7C12_Blinded_Automated_Response_Level_Scoring_and_Freeze.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM AUTHORIZATION
Cell 7C11R manifest SHA-256                   : 630e5df5e9c766ab930ec5af11198457a24427b71954be50193fb6ac7df1d92d
Cell 7C11R terminal PASS verified             : YES
Corrected deterministic input V2              : exact SHA + sidecar verified
A004 endpoint specification                   : exact SHA + sidecar verified
\nBLINDED RESPONSE-LEVEL SCORING
Frozen response observations                  : 1,440
Primary questions                             : 80
Responses per ques